# Lesson 05 Lab — A KV-Cache Memory Budget

**Puzzle:** How many concurrent long-context requests fit after model weights and runtime reserve?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A model can load successfully and still fail when context accumulates. Capacity planning must reserve space for weights, runtime workspace, non-torch allocations, and uncertainty before assigning the remainder to KV cache.


## 0. Predict before running

1. Derive BF16 KV bytes per token for the local checkpoint.
2. Predict the capacity change from BF16 to FP8 cache.
3. State why theoretical concurrency exceeds an operational limit.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The notebook reads the local model configuration and real GPU memory, derives KV bytes per token from layer/head geometry, and computes conservative concurrency for several context lengths and cache dtypes.

- KV geometry comes from the model config.
- Context and concurrency multiply the token footprint.
- A safe budget subtracts weights, workspace, and headroom before division.


## 2. Derive the mechanism

For grouped-query attention, one token stores keys and values for `num_key_value_heads`, not all query heads. A first-order decoder cache uses `2 × layers × kv_heads × head_dim × element_bytes` per token. Dividing a declared KV budget by that footprint gives token capacity; dividing again by context length gives only a theoretical concurrency ceiling.

### Mechanism at a glance

```mermaid
flowchart LR
  G["GPU memory"] --> S["subtract weights"]
  S --> W["subtract workspace + headroom"]
  W --> K["KV budget"]
  M["layers × KV heads × head dim × dtype"] --> B["bytes per token"]
  K --> C["token capacity"]
  B --> C
  C --> R["context × concurrency ceiling"]
```

### Walk it step by step

1. **Read model geometry.** Use KV heads and head dimension, not parameter count alone.
2. **Declare non-KV reserves.** Weights and workspace leave only part of VRAM for context state.
3. **Calculate token capacity.** Divide available bytes by the per-token footprint.
4. **Validate below the ceiling.** Native allocation and latency tests determine the operational limit.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 5
LESSON_TITLE = 'A KV-Cache Memory Budget'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260817
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | BF16 KV storage under one fixed memory budget |
| Candidate | FP8 KV storage and multiple context lengths |
| Held constant | model config, GPU total, weight estimate, reserve fraction, and utilization cap |
| Measurements | KV bytes/token, token capacity, and theoretical concurrent sequences |
| Evidence | `capacity-model` |

**Experiment:** Combine measured GPU memory with model geometry and a declared reserve to calculate context/concurrency cells.


## 5. Inspect the experiment code

The code parses only local configuration fields, shows every subtraction, and emits the full capacity table. No hidden allocator efficiency is inserted into the result.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); layers=int(cfg["num_hidden_layers"]); hidden=int(cfg["hidden_size"])
heads=int(cfg["num_attention_heads"]); kv_heads=int(cfg.get("num_key_value_heads",heads))
head_dim=int(cfg.get("head_dim",hidden//heads)); total=int(torch.cuda.get_device_properties(0).total_memory)
weight_bytes=sum(p.stat().st_size for p in MODEL.glob("*.safetensors")); reserve=3*2**30
usable=max(0,int(total*.85-weight_bytes-reserve))
per_token={"bf16":2*layers*kv_heads*head_dim*2,"fp8":2*layers*kv_heads*head_dim}
capacity={name:usable//size for name,size in per_token.items()}; contexts=(2048,4096,8192,16384)
concurrency={name:{str(ctx):int(tokens//ctx) for ctx in contexts} for name,tokens in capacity.items()}
metrics={"gpu_total_mib":total/2**20,"weight_file_bytes":weight_bytes,"reserve_bytes":reserve,
         "kv_budget_bytes":usable,"geometry":{"layers":layers,"kv_heads":kv_heads,"head_dim":head_dim},
         "kv_bytes_per_token":per_token,"token_capacity":capacity,"concurrency":concurrency}
analysis=(f"Model geometry yields {per_token['bf16']:,} BF16 and {per_token['fp8']:,} FP8 KV "
          f"bytes/token. The declared budget gives a BF16 8K ceiling of {concurrency['bf16']['8192']} "
          "sequences; native allocation and latency must set the operational limit.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| GPU total | 32,110.938 MiB |
| BF16 KV bytes/token | 28,672 bytes |
| FP8 KV bytes/token | 14,336 bytes |
| BF16 token capacity | 778,161 |
| FP8 token capacity | 1,556,323 |
| BF16 8K concurrency | 94 |


## 7. Explain the result

Model geometry yields 28,672 BF16 and 14,336 FP8 KV bytes/token. The declared budget gives a BF16 8K ceiling of 94 sequences; native allocation and latency must set the operational limit.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. Measured environment facts feed explicit planning arithmetic. Assumed topology, demand, bandwidth, and reserve fields remain assumptions until a native deployment test.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 5, "title": 'A KV-Cache Memory Budget', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'KV capacity is a budget equation anchored in model geometry; its result is a planning ceiling until native concurrency tests pass.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 5,
  "title": "A KV-Cache Memory Budget",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260817
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "gpu_total_mib": 32110.9375,
    "weight_file_bytes": 3087467144,
    "reserve_bytes": 3221225472,
    "kv_budget_bytes": 22311452024,
    "geometry": {
      "layers": 28,
      "kv_heads": 2,
      "head_dim": 128
    },
    "kv_bytes_per_token": {
      "bf16": 28672,
      "fp8": 14336
    },
    "token_capacity": {
      "bf16": 778161,
      "fp8": 1556323
    },
    "concurrency": {
      "bf16": {
        "2048": 379,
        "4096": 189,
        "8192": 94,
        "16384": 47
      },
      "fp8": {
        "2048": 759,
        "4096": 379,
        "8192": 189,
        "16384": 94
      }
    }
  },
  "anal

## 9. Make the bounded decision

> KV capacity is a budget equation anchored in model geometry; its result is a planning ceiling until native concurrency tests pass.

**Acceptance/rollback:** Set admission limits below the calculated ceiling and validate them with native load, fragmentation, and tail-latency tests.

**Failure analysis:** Sliding-window attention, hybrid state-space layers, cache alignment, CUDA graphs, prefix sharing, and engine reservations can change the native allocation. Model-file bytes are not identical to resident weight memory.


## 10. Extend the evidence

Start the engine at selected utilization limits, issue long-context concurrency sweeps, and reconcile engine cache-block metrics with the first-order ledger.

The full boundary and references are in [`README.md`](README.md).
